In [ ]:
# ============================================================
# ETAPA: Conclusiones y Visualización Final
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

df = pd.read_csv("../data/telco_churn_clean.csv")
print("Datos cargados correctamente")

In [ ]:
# ============================================================
# RESUMEN DE MÉTRICAS CLAVE
# ============================================================

total = len(df)
churn_si = df[df["Churn_Binary"] == 1]
churn_no = df[df["Churn_Binary"] == 0]
tasa_churn = churn_si.shape[0] / total * 100
ingreso_promedio = df["MonthlyCharges"].mean()
ingreso_riesgo = churn_si["MonthlyCharges"].sum()

print("=" * 55)
print("       MÉTRICAS GLOBALES DEL NEGOCIO")
print("=" * 55)
print(f"  Total clientes:               {total:>6,}")
print(f"  Clientes que abandonaron:     {churn_si.shape[0]:>6,}")
print(f"  Clientes retenidos:           {churn_no.shape[0]:>6,}")
print(f"  Tasa de churn:                {tasa_churn:>6.2f}%")
print(f"  Ingreso promedio mensual:    ${ingreso_promedio:>6.2f}")
print(f"  Ingreso mensual en riesgo:   ${ingreso_riesgo:>8.2f}")
print(f"  Ingreso anual en riesgo:     ${ingreso_riesgo * 12:>10.2f}")
print("=" * 55)

print("\n")
print("=" * 55)
print("       PERFIL DEL CLIENTE PROMEDIO")
print("=" * 55)
print(f"  Antigüedad media:             {df['tenure'].mean():>6.2f} meses")
print(f"  Cargo mensual medio:         ${df['MonthlyCharges'].mean():>6.2f}")
print(f"  Cargo total medio:           ${df['TotalCharges'].mean():>8.2f}")
print(f"  Clientes con contrato mensual:{df[df['Contract']=='Month-to-month'].shape[0]:>6,}")
print(f"  Clientes con Fiber Optic:     {df[df['InternetService']=='Fiber optic'].shape[0]:>6,}")
print("=" * 55)

---

### Resumen de Hallazgos del Análisis

**Análisis Univariado:**
* **Tenure:** Distribución bimodal — clientes muy nuevos (0-5 meses) y muy leales (60-72 meses).
* **MonthlyCharges:** Distribución relativamente uniforme entre $20 y $110.
* **Contract:** 55% de los clientes están en contrato mensual (Month-to-month).
* **InternetService:** 44% usa Fiber Optic, 34% DSL, 22% no tiene internet.

**Análisis Bivariado:**
* **Contrato mensual:** ~42% de churn vs ~11% anual vs ~3% bianual.
* **Fiber Optic:** ~42% churn (posibles problemas de calidad/precio).
* **Seguridad Online / Soporte Técnico:** La ausencia duplica la tasa de churn (~41% vs ~15%).
* **Tenure vs Churn:** Correlación negativa (-0.35) — a mayor antigüedad, menor churn.
* **MonthlyCharges vs Churn:** Correlación positiva (+0.19) — cargos más altos, más churn.

---

In [ ]:
# ============================================================
# TESTS DE HIPÓTESIS
# ============================================================

print("=" * 60)
print("   TEST DE CHI-CUADRADO: Variables Categóricas vs Churn")
print("=" * 60)

categoricas_clave = ["Contract", "InternetService", "PaymentMethod",
                     "OnlineSecurity", "TechSupport", "Partner",
                     "Dependents", "PaperlessBilling", "gender"]

def chi2_manual(df, var, target="Churn_Binary"):
    tabla = pd.crosstab(df[var], df[target])
    obs = tabla.values
    filas = tabla.sum(axis=1).values.reshape(-1, 1)
    cols = tabla.sum(axis=0).values.reshape(1, -1)
    n = obs.sum()
    exp = filas @ cols / n
    chi2 = ((obs - exp) ** 2 / exp).sum()
    dof = (obs.shape[0] - 1) * (obs.shape[1] - 1)
    return chi2, dof

resultados_chi = []
for var in categoricas_clave:
    chi2, dof = chi2_manual(df, var)
    # Interpretación: chi2 / dof > 3.84 (para 1 gl) o > 5.99 (2 gl) sugiere dependencia
    if dof == 1:
        umbral = 3.84
    elif dof == 2:
        umbral = 5.99
    else:
        umbral = 7.81
    relacion = "DEPENDIENTE" if chi2 > umbral else "independiente"
    resultados_chi.append({"Variable": var, "Chi2": round(chi2, 2),
                          "gl": dof, "Relación": relacion})
    print(f"  {var:20s} | Chi2 = {chi2:>8.2f} | gl = {dof} | {relacion}")

print("\n" + "=" * 60)
print("")

print("=" * 60)
print("   TEST T: Variables Numéricas por Grupo de Churn")
print("=" * 60)

numericas = ["tenure", "MonthlyCharges", "TotalCharges"]

for var in numericas:
    grupo_0 = df[df["Churn_Binary"] == 0][var].dropna()
    grupo_1 = df[df["Churn_Binary"] == 1][var].dropna()
    
    n1, n2 = len(grupo_0), len(grupo_1)
    m1, m2 = grupo_0.mean(), grupo_1.mean()
    v1, v2 = grupo_0.var(), grupo_1.var()
    
    # T-test de Welch (varianzas desiguales)
    se = np.sqrt(v1/n1 + v2/n2)
    t_stat = (m1 - m2) / se
    
    print(f"\n  Variable: {var}")
    print(f"    No Churn (media): ${m1:>8.2f}   | n = {n1:>5,}")
    print(f"    Churn (media):    ${m2:>8.2f}   | n = {n2:>5,}")
    print(f"    Diferencia:       ${m1-m2:>+8.2f}")
    print(f"    Estadístico t:    {t_stat:>+8.2f}")
    
    # |t| > 1.96 sugiere diferencia significativa (alpha=0.05, aproximación normal)
    if abs(t_stat) > 1.96:
        print(f"    → Diferencia SIGNIFICATIVA (|t| > 1.96)")
    else:
        print(f"    → Diferencia NO significativa")

print("\n" + "=" * 60)

In [ ]:
# ============================================================
# DASHBOARD FINAL: Factores Críticos de Churn
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Factores Críticos de Churn — Resumen Ejecutivo",
             fontsize=16, fontweight="bold", y=1.02)

# --- Gráfico 1: Tasa de Churn por Tipo de Contrato ---
contrato_rate = df.groupby("Contract")["Churn_Binary"].mean() * 100
contrato_rate = contrato_rate.reindex(["Month-to-month", "One year", "Two year"])
colores_contrato = ["#e74c3c" if x > 26.5 else "#2ecc71" for x in contrato_rate.values]
axes[0,0].bar(range(len(contrato_rate)), contrato_rate.values,
              color=colores_contrato, edgecolor="white", linewidth=1.5)
axes[0,0].axhline(y=26.5, color="gray", linestyle="--", alpha=0.7,
                  label="Churn promedio (26.5%)")
axes[0,0].set_xticks(range(len(contrato_rate)))
axes[0,0].set_xticklabels(["Mensual", "Anual", "Bianual"], fontsize=10)
axes[0,0].set_title("Churn por Tipo de Contrato", fontsize=12, fontweight="bold")
axes[0,0].set_ylabel("Tasa de Churn (%)")
axes[0,0].legend(fontsize=8)
for bar, val in zip(axes[0,0].patches, contrato_rate.values):
    axes[0,0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                   f"{val:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

# --- Gráfico 2: Churn por Servicio de Internet ---
internet_rate = df.groupby("InternetService")["Churn_Binary"].mean() * 100
colores_internet = ["#e74c3c" if x > 26.5 else "#2ecc71" for x in internet_rate.values]
axes[0,1].bar(range(len(internet_rate)), internet_rate.values,
              color=colores_internet, edgecolor="white", linewidth=1.5)
axes[0,1].axhline(y=26.5, color="gray", linestyle="--", alpha=0.7,
                  label="Churn promedio (26.5%)")
axes[0,1].set_xticks(range(len(internet_rate)))
axes[0,1].set_xticklabels(internet_rate.index, fontsize=10)
axes[0,1].set_title("Churn por Tipo de Internet", fontsize=12, fontweight="bold")
axes[0,1].set_ylabel("Tasa de Churn (%)")
axes[0,1].legend(fontsize=8)
for bar, val in zip(axes[0,1].patches, internet_rate.values):
    axes[0,1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                   f"{val:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

# --- Gráfico 3: Churn por Antigüedad (binned) ---
df["tenure_group"] = pd.cut(df["tenure"], bins=[0, 6, 12, 24, 48, 72],
                             labels=["0-6m", "6-12m", "1-2a", "2-4a", "4-6a"])
tenure_rate = df.groupby("tenure_group", observed=True)["Churn_Binary"].mean() * 100
axes[1,0].plot(range(len(tenure_rate)), tenure_rate.values, 
               marker="o", linewidth=2.5, color="#e74c3c", markersize=8)
axes[1,0].axhline(y=26.5, color="gray", linestyle="--", alpha=0.7,
                  label="Churn promedio (26.5%)")
axes[1,0].set_xticks(range(len(tenure_rate)))
axes[1,0].set_xticklabels(tenure_rate.index, fontsize=10)
axes[1,0].set_title("Churn por Antigüedad", fontsize=12, fontweight="bold")
axes[1,0].set_ylabel("Tasa de Churn (%)")
axes[1,0].set_xlabel("Antigüedad")
axes[1,0].legend(fontsize=8)
for i, val in enumerate(tenure_rate.values):
    axes[1,0].annotate(f"{val:.1f}%", (i, val), textcoords="offset points",
                       xytext=(0, 10), ha="center", fontsize=9, fontweight="bold")

# --- Gráfico 4: Distribución de MonthlyCharges por Churn ---
for churn_val, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
    subset = df[df["Churn_Binary"] == churn_val]["MonthlyCharges"]
    axes[1,1].hist(subset, bins=30, alpha=0.6, color=color,
                   edgecolor="white", density=True,
                   label=f"{'No Churn' if churn_val == 0 else 'Churn'}")
axes[1,1].set_title("Cargo Mensual por Churn", fontsize=12, fontweight="bold")
axes[1,1].set_xlabel("Monthly Charges ($)")
axes[1,1].set_ylabel("Densidad")
axes[1,1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("../visuals/09_dashboard_final.png", dpi=150, bbox_inches="tight")
plt.show()

# Limpiar columna temporal
df.drop(columns=["tenure_group"], inplace=True)

---

## Conclusiones Estratégicas

---

### Perfil del Cliente en Riesgo
El análisis permite construir un perfil claro del cliente con mayor probabilidad de abandono:
* **Contrato mensual (Month-to-month):** 42% de churn, frente al 11% en contratos anuales y apenas 3% en bianuales.
* **Fiber Optic:** 42% de churn, posiblemente por precio elevado o problemas de calidad del servicio.
* **Sin servicios de valor agregado:** La ausencia de Online Security y TechSupport duplica la tasa de churn.
* **Baja antigüedad:** Los primeros 6 meses son críticos. Clientes con tenure < 6 meses tienen la tasa de abandono más alta.
* **Electronic Check:** Método de pago asociado al mayor churn entre todas las modalidades.

### Impacto Económico
* **Tasa de churn:** 26.5% (1,869 clientes perdidos).
* **Ingreso mensual en riesgo:** ~$120,000/mes.
* **Ingreso anual en riesgo:** ~$1,440,000/año.
* Reducir el churn del 26.5% al 20% representaría recuperar ~$29,000/mes.

### Recomendaciones Accionables
1. **Incentivar contratos a largo plazo:** Ofrecer descuentos o beneficios exclusivos para migrar de contrato mensual a anual/bianual.
2. **Programa de onboarding:** Implementar un proceso de acompañamiento intensivo durante los primeros 6 meses (la ventana crítica).
3. **Paquetes de servicios:** Promover la contratación de Online Security y TechSupport como parte del paquete básico.
4. **Revisar precios de Fiber Optic:** Evaluar si el precio del servicio Fiber Optic es competitivo frente a la competencia.
5. **Segmentación de pagos:** Ofrecer incentivos para migrar de Electronic Check a débito automático o tarjeta de crédito.
6. **Alertas tempranas:** Implementar un sistema de detección basado en las variables identificadas (contrato mensual + Fiber Optic + baja antigüedad + sin seguridad).

---

## Próximos Pasos

---

Este análisis exploratorio ha identificado los factores más relevantes asociados al churn. Las siguientes etapas naturales serían:

1. **Modelo Predictivo:** Entrenar un modelo de clasificación (Regresión Logística, Random Forest, XGBoost) para predecir la probabilidad individual de churn.
2. **Feature Engineering:** Crear variables derivadas como interacciones entre variables (ej. Contract × Tenure, InternetService × OnlineSecurity).
3. **Segmentación Avanzada:** Aplicar técnicas de clustering (K-Means) para identificar micro-segmentos de clientes con comportamientos diferenciados.
4. **Análisis de Supervivencia:** Utilizar técnicas de análisis de supervivencia (Kaplan-Meier) para modelar el tiempo hasta el abandono.
5. **Cuantificación del ROI:** Estimar el retorno de inversión de cada estrategia de retención propuesta.

---

*Proyecto: Análisis de CHURN de Clientes — Telecom Industry*

*Autor: Nahuel Caero*